In [1]:
pip install -q langchain-groq langchain-core requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
import requests

In [3]:
# tool 

@tool
def multiply(a: int, b: int) -> int:
    """ Given 2 numbers a and b this tool return their product """
    return a*b

In [4]:
print(multiply.invoke({'a':3,'b':4}))

12


In [5]:
multiply.name

'multiply'

In [6]:
multiply.description

'Given 2 numbers a and b this tool return their product'

In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [11]:
llm_with_tools = llm.bind_tools([multiply])

In [12]:
query = HumanMessage("can u multiply 3 with 10?")

In [13]:
message = [query]

In [14]:
message

[HumanMessage(content='can u multiply 3 with 10?', additional_kwargs={}, response_metadata={})]

In [15]:
result = llm_with_tools.invoke(message)

In [16]:
message.append(result)

In [17]:
message

[HumanMessage(content='can u multiply 3 with 10?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9zp80gxqd', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 239, 'total_tokens': 258, 'completion_time': 0.055927836, 'completion_tokens_details': None, 'prompt_time': 0.060407639, 'prompt_tokens_details': None, 'queue_time': 0.16251475, 'total_time': 0.116335475}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d9769-9d8d-7863-96fc-cf4be542df0e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': '9zp80gxqd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 19, 'total_tokens': 258})]

In [18]:
tool_result = multiply.invoke(result.tool_calls[0]["args"])

In [19]:
#message.append(tool_result)
message.append(ToolMessage(
        content=str(tool_result),
        tool_call_id=result.tool_calls[0]["id"]
    )
)

In [20]:
message

[HumanMessage(content='can u multiply 3 with 10?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9zp80gxqd', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 239, 'total_tokens': 258, 'completion_time': 0.055927836, 'completion_tokens_details': None, 'prompt_time': 0.060407639, 'prompt_tokens_details': None, 'queue_time': 0.16251475, 'total_time': 0.116335475}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d9769-9d8d-7863-96fc-cf4be542df0e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': '9zp80gxqd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 19, 'total_tokens': 258}),
 ToolMessage

In [21]:
llm_with_tools.invoke(message).content

'The result of multiplying 3 by 10 is 30.'

In [22]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency:str, target_currency:str) -> float:
    """
    this function fetch the currency conversion factor between a given base currency and a target currenct
    """

    url = f'https://v6.exchangerate-api.com/v6/93f0cf38ea5cc9d54abbbf57/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()


@tool
def convert(base_currency_value: int, conversion_rate:Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [23]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776297602,
 'time_last_update_utc': 'Thu, 16 Apr 2026 00:00:02 +0000',
 'time_next_update_unix': 1776384002,
 'time_next_update_utc': 'Fri, 17 Apr 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 93.4415}

In [24]:
convert.invoke({'base_currency_value':10,'conversion_rate':92.9583})

929.583

In [25]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [26]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [27]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [28]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [29]:
ai_message = llm_with_tools.invoke(messages)

In [30]:
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ztyrpwpgh', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': '5m55y4nfd', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 354, 'total_tokens': 392, 'completion_time': 0.083032953, 'completion_tokens_details': None, 'prompt_time': 0.03402533, 'prompt_tokens_details': None, 'queue_time': 0.159223387, 'total_time': 0.117058283}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d9769-bd72-7361-b558-ec003f4b3f33-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'ztyrpwpgh', 'type': 'tool_call'}, {'nam

In [31]:
messages.append(ai_message)

In [32]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'ztyrpwpgh',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '5m55y4nfd',
  'type': 'tool_call'}]

In [33]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [34]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ztyrpwpgh', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': '5m55y4nfd', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 354, 'total_tokens': 392, 'completion_time': 0.083032953, 'completion_tokens_details': None, 'prompt_time': 0.03402533, 'prompt_tokens_details': None, 'queue_time': 0.159223387, 'total_time': 0.117058283}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d9769-bd72-7361-b558

In [35]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 83.4415. Based on this conversion factor, 10 USD is equivalent to 834.415 INR.'